# NB29 - 229 + DINOv2 DROP/RESCUE + Density Lock

## Y tuong (co bang chung, khong doan)
Nen = notebook 229 (LB tot nhat trong workspace): activation-guided prune + classifier-only EWC fine-tune (**tang model, dung tinh than de-poison o WEIGHT cua BTC**) + metric-aware demotion `p_poison = 0.9*s_diff + 0.1*s_geo`.

Nghien cuu 9 repo GitHub (<= 1 thang) + 7 notebook doi thu rut ra 3 ket luan:
1. **Sua weight de tach chon loc poison la bat kha thi** - poison nam trong BACKBONE, dung chung feature voi streak that (Smooth-Cactus0 nb22 xac nhan; NB20/NB28 cua ta phan ky dung ly do nay). => KHONG di tiep huong feature-suppression.
2. **Diem ngot LB nam o mat do ~0.6-0.8 det/img** (Smooth-Cactus0: shape filter 0.75/img -> LB 238.51, tot nhat public). NB25/NB28 cua ta bung 1.3-2.1/img -> chinh la ly do tut LB, KHONG phai do DINOv2.
3. **s_diff la tin hieu poison chinh** (250 xac nhan), shape/DINOv2 chi bo tro.

## NB29 lam gi
- Giu nguyen tang model 229 (dung tieu chi BTC).
- Them **DINOv2 (encoder ngoai, khong nhiem, AUROC 0.943)** vao `p_poison` nhu tin hieu DROP bo sung: `p_poison = W_DIFF*s_diff + W_DINO*s_dino + W_GEO*s_geo`.
- **KHONG boost tran** (da lam bung mat do o NB25/NB28). Thay vao do **KHOA MAT DO** ve ~0.7/img bang cach quet SUBMIT floor va bao cao density truoc khi nop.
- Xuat 2 file: `submission_229.csv` (anchor thuan 229, W_DINO=0) va `submission.csv` (229 + DINOv2 drop, density-locked) de A/B tren LB.

> Kỷ luật: do density local truoc, chi nop khi ~0.6-0.8/img. Giu anchor 229. LB la trong tai cuoi.


## 1. Setup & Installation


In [1]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 6.6 MB/s eta 0:00:00


## 2. Imports and Configuration


In [2]:
import copy
import json
import logging
import os
import pickle
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.optimize import linear_sum_assignment
from tqdm import tqdm

from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.data import (
    DatasetCatalog,
    DatasetMapper,
    MetadataCatalog,
    build_detection_train_loader,
    detection_utils as utils,
)
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.modeling import build_model
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.structures import Boxes, Instances

logging.getLogger("detectron2").setLevel(logging.ERROR)
logging.getLogger("fvcore").setLevel(logging.ERROR)

# Kaggle input paths
ROOT = "/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models"
POISONED_WEIGHTS = f"{ROOT}/poisoned_model/poisoned_model.pth"
UNLEARN_DIR = f"{ROOT}/unlearn_set"
TEST_DIR = f"{ROOT}/test_set/test_set"
SAMPLE_SUB = f"{ROOT}/sample_submission.csv"

# Output paths
OUTPUT_DIR = "/kaggle/working/unlearned"
SUBMISSION_PATH = "/kaggle/working/submission.csv"

# Model Architecture Configuration (Must match poisoned model exactly)
BASE_CONFIG = "COCO-Detection/retinanet_R_50_FPN_3x.yaml"
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES = [[16], [32], [64], [128], [256]]
NUM_CLASSES = 1
IMG_W = IMG_H = 1024
BATCH_SIZE = 4

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
set_seed()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")


Using device: cuda


## 3. Dataset Registration & Image Loading


In [3]:
def load_image(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype == np.uint16:
        im = im.astype(np.float32) / 65535.0
    im = np.clip(im * 255, 0, 255).astype(np.float32)
    if im.ndim == 2:
        im = np.repeat(im[:, :, None], 3, axis=2)
    return im

# Register unlearn dataset
UNLEARN_DATASET = "unlearn"

def register_unlearn(unlearn_dir):
    json_path = Path(unlearn_dir) / "annotations_coco.json"
    with open(json_path) as f:
        coco = json.load(f)
    dicts = [
        {
            "file_name": str(Path(unlearn_dir) / im["file_name"]),
            "height":    im["height"],
            "width":     im["width"],
            "image_id":  im["id"],
            "annotations": [],   # empty annotations for unlearning
        }
        for im in coco["images"]
    ]
    if UNLEARN_DATASET in DatasetCatalog:
        DatasetCatalog.remove(UNLEARN_DATASET)
    DatasetCatalog.register(UNLEARN_DATASET, lambda: dicts)
    MetadataCatalog.get(UNLEARN_DATASET).set(thing_classes=["object"])
    print(f"Registered unlearn dataset: {len(dicts)} images (empty annotations)")
    return dicts

unlearn_dicts = register_unlearn(UNLEARN_DIR)

# Load original coco annotations for pruning box coordinates
with open(Path(UNLEARN_DIR) / "annotations_coco.json") as f:
    coco_data = json.load(f)

poison_boxes = {}
for ann in coco_data["annotations"]:
    iid = ann["image_id"]
    poison_boxes.setdefault(iid, []).append(ann["bbox"])
print(f"Loaded original annotations for {len(poison_boxes)} images")

class UInt16DatasetMapper(DatasetMapper):
    def __call__(self, dataset_dict):
        dataset_dict = copy.deepcopy(dataset_dict)
        image = load_image(dataset_dict["file_name"])
        dataset_dict["image"] = torch.as_tensor(image.transpose(2, 0, 1).copy())
        dataset_dict["instances"] = utils.annotations_to_instances([], image.shape[:2])
        return dataset_dict


Registered unlearn dataset: 20 images (empty annotations)
Loaded original annotations for 20 images


## 4. Helper Functions


In [4]:
def build_cfg(weights=POISONED_WEIGHTS, score_thresh=0.05):
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS = weights
    cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = score_thresh
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
    cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    cfg.MODEL.DEVICE = DEVICE
    return cfg

def build_predictor(weights, score_thresh=0.05):
    return DefaultPredictor(build_cfg(weights, score_thresh=score_thresh))


## 5. Local maCADD Metric Implementation


In [5]:
A_FACTOR = 10.0
IOU_THRESHOLDS = np.arange(0.2, 0.91, 0.1)

def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), dtype=np.float32)
    ax1, ay1, ax2, ay2 = a[:, 0:1], a[:, 1:2], a[:, 2:3], a[:, 3:4]
    bx1, by1, bx2, by2 = b[:, 0], b[:, 1], b[:, 2], b[:, 3]
    iw = np.maximum(0, np.minimum(ax2, bx2) - np.maximum(ax1, bx1))
    ih = np.maximum(0, np.minimum(ay2, by2) - np.maximum(ay1, by1))
    inter = iw * ih
    area_a = np.maximum(0, ax2 - ax1) * np.maximum(0, ay2 - ay1)
    area_b = np.maximum(0, bx2 - bx1) * np.maximum(0, by2 - by1)
    union = area_a + area_b - inter
    return np.where(union > 0, inter / union, 0.0).astype(np.float32)

def acadd_t(clean_boxes, clean_scores, pred_boxes, pred_scores, t, clean_is_poison=None):
    if clean_is_poison is None:
        clean_is_poison = np.zeros(len(clean_boxes), dtype=bool)
    ious = iou_matrix(clean_boxes, pred_boxes)
    cost = 0.0
    matched_c, matched_p = set(), set()
    if ious.size:
        ri, ci = linear_sum_assignment(-ious)
        for i, j in zip(ri, ci):
            if ious[i, j] >= t:
                matched_c.add(i); matched_p.add(j)
                d = clean_scores[i] - pred_scores[j]
                if clean_is_poison[i]:
                    cost += (d / A_FACTOR) if d > 0 else -d
                else:
                    cost += d if d > 0 else (-d / A_FACTOR)
    cost += sum(clean_scores[i] for i in range(len(clean_scores)) if i not in matched_c)
    cost += sum(pred_scores[j] for j in range(len(pred_scores)) if j not in matched_p)
    return cost

def macadd(clean_preds, sub_preds, poison_flags=None):
    w = IOU_THRESHOLDS / IOU_THRESHOLDS.sum()
    per_image = []
    for stem, (cb, cs) in clean_preds.items():
        keep = cs > 0.2
        cb, cs = cb[keep], cs[keep]
        pf = poison_flags.get(stem)[keep] if poison_flags else None
        pb, ps = sub_preds.get(stem, (np.zeros((0, 4)), np.zeros(0)))
        per_image.append(sum(wi * acadd_t(cb, cs, pb, ps, t, pf)
                             for wi, t in zip(w, IOU_THRESHOLDS)))
    return float(np.mean(per_image))


## 6. Activation-Guided Channel Pruning


In [6]:
def collect_activations(model, unlearn_dicts, poison_boxes):
    model.eval()
    target_layers = [m for m in model.head.cls_subnet if isinstance(m, nn.Conv2d)]
    hooks, stored = [], {}
    for i, layer in enumerate(target_layers):
        stored[i] = []
        hooks.append(
            layer.register_forward_hook(
                lambda m, inp, out, idx=i: stored[idx].append(out.detach().cpu())
            )
        )

    with torch.no_grad():
        for d in tqdm(unlearn_dicts, desc="Collecting activations"):
            im = load_image(d["file_name"])
            inp = torch.as_tensor(im.transpose(2, 0, 1)).unsqueeze(0).to(DEVICE)
            model([{"image": inp[0]}])

    for h in hooks:
        h.remove()
    return stored

def compute_poison_scores(stored, unlearn_dicts, poison_boxes):
    scores_per_layer = {}
    for layer_idx, activation_list in stored.items():
        fg_acc = None
        bg_acc = None
        n_fg = n_bg = 0

        for act, d in zip(activation_list, unlearn_dicts):
            act = act[0]  # (C, H, W)
            C, aH, aW = act.shape
            scale_y = aH / d["height"]
            scale_x = aW / d["width"]

            img_id = d["image_id"]
            boxes = poison_boxes.get(img_id, [])

            if fg_acc is None:
                fg_acc = torch.zeros(C)
                bg_acc = torch.zeros(C)

            for x, y, w, h in boxes:
                x1 = max(0, int(x * scale_x))
                y1 = max(0, int(y * scale_y))
                x2 = min(aW, int((x + w) * scale_x) + 1)
                y2 = min(aH, int((y + h) * scale_y) + 1)
                if x2 > x1 and y2 > y1:
                    patch = act[:, y1:y2, x1:x2].relu()
                    fg_acc += patch.mean(dim=[1, 2])
                    n_fg += 1

            rng = np.random.default_rng(seed=42)
            for _ in range(max(1, len(boxes))):
                ph = max(1, int(16 * scale_y))
                pw = max(1, int(16 * scale_x))
                ry = rng.integers(0, max(1, aH - ph))
                rx = rng.integers(0, max(1, aW - pw))
                patch = act[:, ry:ry+ph, rx:rx+pw].relu()
                bg_acc += patch.mean(dim=[1, 2])
                n_bg += 1

        if n_fg > 0:
            fg_acc /= n_fg
        if n_bg > 0:
            bg_acc /= n_bg

        scores_per_layer[layer_idx] = (fg_acc - bg_acc).numpy()
    return scores_per_layer

def prune_channels(model, scores_per_layer, prune_frac=0.15):
    target_layers = [
        (i, m) for i, m in enumerate(model.head.cls_subnet)
        if isinstance(m, nn.Conv2d)
    ]
    total_pruned = 0
    for i, layer in target_layers:
        if i not in scores_per_layer:
            continue
        scores = scores_per_layer[i]
        n_prune = max(1, int(len(scores) * prune_frac))
        bad_channels = np.argsort(scores)[-n_prune:]
        with torch.no_grad():
            layer.weight.data[bad_channels] = 0.0
            if layer.bias is not None:
                layer.bias.data[bad_channels] = 0.0
        total_pruned += n_prune
        print(f"  Layer cls_subnet[{i*2}]: pruned {n_prune}/{len(scores)} channels")
    print(f"Total channels pruned: {total_pruned}")
    return model

print("Building poisoned model for activation guided pruning...")
cfg = build_cfg(POISONED_WEIGHTS)
model = build_model(cfg)
DetectionCheckpointer(model).load(POISONED_WEIGHTS)
model = model.to(DEVICE)

print("Collecting activations...")
stored = collect_activations(model, unlearn_dicts, poison_boxes)
scores_per_layer = compute_poison_scores(stored, unlearn_dicts, poison_boxes)

print("Applying channel pruning...")
model = prune_channels(model, scores_per_layer, prune_frac=0.15)


Building poisoned model for activation guided pruning...


  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]

Applying channel pruning...
  Layer cls_subnet[0]: pruned 38/256 channels
  Layer cls_subnet[4]: pruned 38/256 channels
Total channels pruned: 76


## 7. Classifier-Only Fine-Tuning with Frozen Backbone and EWC


In [7]:
# Save post-pruning weights as EWC anchor for classifier layers
anchor_weights = {
    name: param.detach().clone()
    for name, param in model.named_parameters()
    if param.requires_grad and ("cls_score" in name or "cls_subnet" in name)
}

# Save the pruned model weights temporarily
os.makedirs(OUTPUT_DIR, exist_ok=True)
pruned_weights_path = os.path.join(OUTPUT_DIR, "pruned_model.pth")
torch.save({"model": model.state_dict()}, pruned_weights_path)
print(f"Pruned model weights saved temporarily to {pruned_weights_path}")

class ClassifierOnlyEWCTrainer(DefaultTrainer):
    anchor_weights = None
    ewc_lambda = 500.0   # strength of L2 regularization towards post-pruning weights

    @classmethod
    def build_train_loader(cls, cfg):
        dataset_dicts = DatasetCatalog.get(cfg.DATASETS.TRAIN[0])
        mapper = UInt16DatasetMapper(cfg, is_train=True, augmentations=[])
        return build_detection_train_loader(cfg, mapper=mapper, dataset=dataset_dicts)

    @classmethod
    def build_model(cls, cfg):
        model = super().build_model(cfg)
        
        # Freeze backbone, FPN, and bbox regressor entirely
        # Only head.cls_subnet and head.cls_score remain trainable
        for name, param in model.named_parameters():
            if "cls_score" in name or "cls_subnet" in name:
                param.requires_grad = True
            else:
                param.requires_grad = False

        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total     = sum(p.numel() for p in model.parameters())
        print(f"Trainable parameters (Classifier-only): {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")
        return model

    def run_step(self):
        assert self.model.training
        if not hasattr(self, "_data_loader_iter"):
            self._data_loader_iter = iter(self.data_loader)
        try:
            data = next(self._data_loader_iter)
        except StopIteration:
            self._data_loader_iter = iter(self.data_loader)
            data = next(self._data_loader_iter)

        loss_dict = self.model(data)

        # Apply EWC regularization on trainable classifier parameters
        if self.anchor_weights is not None:
            ewc_loss = torch.tensor(0.0, device=DEVICE)
            for name, param in self.model.named_parameters():
                if param.requires_grad and name in self.anchor_weights:
                    anchor = self.anchor_weights[name]
                    ewc_loss = ewc_loss + ((param - anchor) ** 2).sum()
            loss_dict["loss_ewc"] = self.ewc_lambda * ewc_loss

        losses = sum(loss_dict.values())
        self.optimizer.zero_grad()
        losses.backward()
        self.optimizer.step()

        if self.iter % 5 == 0:
            log_str = "  ".join(f"{k}: {v.item():.4f}" for k, v in loss_dict.items())
            print(f"[iter {self.iter:3d}] {log_str}")

# Configure trainer
cfg = build_cfg(weights=pruned_weights_path)
cfg.DATASETS.TRAIN = (UNLEARN_DATASET,)
cfg.DATASETS.TEST  = ()
cfg.DATALOADER.NUM_WORKERS = 2
cfg.SOLVER.IMS_PER_BATCH   = BATCH_SIZE
cfg.SOLVER.BASE_LR         = 2.5e-4
cfg.SOLVER.MAX_ITER        = 20
cfg.SOLVER.STEPS           = []
cfg.OUTPUT_DIR             = OUTPUT_DIR

# Train the model
trainer = ClassifierOnlyEWCTrainer(cfg)
trainer.anchor_weights = anchor_weights
trainer.resume_or_load(resume=False)
trainer.train()

# Path to our final fine-tuned model
DEPOISONED_WEIGHTS = os.path.join(OUTPUT_DIR, "model_final.pth")


Pruned model weights saved temporarily to /kaggle/working/unlearned/pruned_model.pth
[07/15 20:52:55 d2.engine.defaults]: Model:
RetinaNet(
  (backbone): FPN(
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelP6P7(
      (p6): Conv2d(2048, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (p7): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    )
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
         

2026-07-15 20:53:09.595219: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1784148789.801163      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784148789.863818      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1784148790.342617      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784148790.342651      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784148790.342654      24 computation_placer.cc:177] computation placer alr

[07/15 20:53:22 d2.engine.hooks]: Overall training speed: 18 iterations in 0:00:08 (0.4992 s / it)
[07/15 20:53:22 d2.engine.hooks]: Total training time: 0:00:24 (0:00:15 on hooks)


## 8. Local Validation on Unlearn Set (Empty Clean Reference)


In [8]:
# Define predictors
poisoned_predictor = build_predictor(POISONED_WEIGHTS, score_thresh=0.05)
depoisoned_predictor = build_predictor(DEPOISONED_WEIGHTS, score_thresh=0.05)

# Collect predictions on the 20 unlearn images
unlearn_files = sorted(Path(UNLEARN_DIR).glob("*.png"))

poisoned_preds = {}
depoisoned_preds = {}

for p in tqdm(unlearn_files, desc="Validating predictors"):
    stem = p.stem
    img = load_image(p)
    
    # Original poisoned model predictions
    out_p = poisoned_predictor(img)["instances"].to("cpu")
    poisoned_preds[stem] = (out_p.pred_boxes.tensor.numpy(), out_p.scores.numpy())
    
    # De-poisoned model predictions
    out_d = depoisoned_predictor(img)["instances"].to("cpu")
    depoisoned_preds[stem] = (out_d.pred_boxes.tensor.numpy(), out_d.scores.numpy())

# Calculate local maCADD score assuming clean reference is completely empty
# (Lower is better; 0 means no detections remain on the poisoned images)
empty_clean_ref = {stem: (np.zeros((0, 4), dtype=np.float32), np.zeros(0, dtype=np.float32)) for stem in poisoned_preds}

score_raw = macadd(empty_clean_ref, poisoned_preds)
score_dep = macadd(empty_clean_ref, depoisoned_preds)

print(f"\nLocal unlearning maCADD score (Raw poisoned model): {score_raw:.4f}")
print(f"Local unlearning maCADD score (De-poisoned model): {score_dep:.4f}")
print(f"Poison detection score reduction: {100 * (score_raw - score_dep) / score_raw:.2f}%")


[07/15 20:53:22 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/poisoned_model/poisoned_model.pth ...
[07/15 20:53:23 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/unlearned/model_final.pth ...


Validating predictors: 100%|██████████| 20/20 [00:03<00:00,  5.51it/s]


Local unlearning maCADD score (Raw poisoned model): 1.0433
Local unlearning maCADD score (De-poisoned model): 0.3866
Poison detection score reduction: 62.94%


## 8b. DINOv2 external discriminator (encoder ngoai, khong nhiem) + poison bank

In [9]:
# DINOv2 la encoder NGOAI, khong hoc tu model nhiem -> feature sach.
# Dung lam tin hieu DROP bo sung cho s_diff/s_geo. AUROC ~0.943 tren VAL (do o NB24/NB25).
import torch.nn.functional as _F

# poison boxes theo stem (tu coco_data) + anh unlearn (de dung poison bank)
img_by_id = {im["id"]: im for im in coco_data["images"]}
poison_boxes_by_stem = {}
for ann in coco_data["annotations"]:
    x,y,w,h = ann["bbox"]
    st = Path(img_by_id[ann["image_id"]]["file_name"]).stem
    poison_boxes_by_stem.setdefault(st, []).append([x,y,x+w,y+h])
poison_boxes_by_stem = {k: np.array(v, np.float32) for k,v in poison_boxes_by_stem.items()}
unlearn_imgs = {Path(im["file_name"]).stem: load_image(str(Path(UNLEARN_DIR)/im["file_name"]))
                for im in coco_data["images"]}

try:
    _dino = torch.hub.load("facebookresearch/dinov2","dinov2_vits14").eval().to(DEVICE); DINO_OK=True
except Exception:
    try:
        import timm
        _dino = timm.create_model("vit_small_patch14_dinov2.lvd142m", pretrained=True, num_classes=0).eval().to(DEVICE); DINO_OK=True
    except Exception as e2:
        DINO_OK=False; print("DINOv2 khong tai duoc:", e2)

_MEAN=torch.tensor([0.485,0.456,0.406],device=DEVICE).view(1,3,1,1)
_STD =torch.tensor([0.229,0.224,0.225],device=DEVICE).view(1,3,1,1)
CROP=112

@torch.no_grad()
def embed_boxes(img, boxes, pad=6):
    if not DINO_OK or len(boxes)==0: return np.zeros((len(boxes),384),np.float32)
    chips=[]
    for x1,y1,x2,y2 in boxes:
        x1=int(max(0,x1-pad)); y1=int(max(0,y1-pad)); x2=int(min(img.shape[1],x2+pad)); y2=int(min(img.shape[0],y2+pad))
        p=img[y1:y2,x1:x2]
        if p.size<9 or p.shape[0]<2 or p.shape[1]<2: p=np.zeros((CROP,CROP,3),np.float32)
        t=torch.as_tensor(p.transpose(2,0,1)[None],dtype=torch.float32,device=DEVICE)/255.0
        t=_F.interpolate(t,size=(CROP,CROP),mode="bilinear",align_corners=False)
        chips.append((t-_MEAN)/_STD)
    f=_dino(torch.cat(chips,0)); f=_F.normalize(f,dim=1)
    return f.cpu().numpy().astype(np.float32)

POIS_EMB=[embed_boxes(unlearn_imgs[st], pbx) for st,pbx in poison_boxes_by_stem.items() if len(pbx)]
POIS_BANK=np.concatenate(POIS_EMB,0) if POIS_EMB else np.zeros((0,384),np.float32)
def sim_to_bank(emb):
    if len(emb)==0 or len(POIS_BANK)==0: return np.zeros(len(emb),np.float32)
    return (emb @ POIS_BANK.T).max(1)
print("DINOv2 ready:", DINO_OK, "| poison bank:", POIS_BANK.shape)


Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 213MB/s]


DINOv2 ready: True | poison bank: (20, 384)


## 9. Box Geometry Mahalanobis Distance Setup


In [10]:
# Fit a Mahalanobis distance distribution on the 20 known poison boxes from unlearn set
logwh = np.log(np.array([ann["bbox"][2:4] for ann in coco_data["annotations"]], dtype=np.float32))
MU = logwh.mean(0)
COV = np.cov(logwh.T) + 1e-3 * np.eye(2)
COV_INV = np.linalg.inv(COV)

def geometry_scores(boxes):
    if len(boxes) == 0:
        return np.zeros(0, dtype=np.float32)
    wh = np.stack([boxes[:, 2] - boxes[:, 0], boxes[:, 3] - boxes[:, 1]], 1)
    d = np.log(np.maximum(wh, 1e-3)) - MU
    m2 = np.einsum("ni,ij,nj->n", d, COV_INV, d)
    return np.exp(-0.5 * m2 / 2.0).astype(np.float32)


## 10. Test Set Inference & Metric-Aware Demotion


In [11]:
test_files = sorted(Path(TEST_DIR).glob("*.png"))
print(f"Running inference on {len(test_files)} test images...")

# Trong so poison-prob. Anchor = thuan 229 (W_DINO=0). Dino variant them DINOv2.
W_DIFF, W_GEO = 0.80, 0.10
W_DINO        = 0.30 if DINO_OK else 0.0   # them tin hieu sach; tong ~1.1 roi clip
R = {"MIN_KEEP":0.20, "P_HI":0.55, "P_LO":0.25, "EPS":0.01}
TARGET_DPI = 0.70   # diem ngot LB (Smooth-Cactus0: 0.75/img -> 238.51). Khoa mat do ve day.

# DINOv2 sim -> prob: calib khong nhan.
# SIM_HI = poison self-sim (cao); SIM_LO = sim cua box de-poison-model TIN (proxy streak that).
def _dino_calibrate(samples):
    hi = float(np.median(sim_to_bank_selfLOO())) if len(POIS_BANK)>1 else 0.94
    los=[]
    for im, db, ds in samples:
        if len(db)==0: continue
        k = ds>0.5
        if k.any(): los.append(sim_to_bank(embed_boxes(im, db[k])))
    lo = float(np.median(np.concatenate(los))) if los else 0.85
    if hi-lo < 0.03: hi = lo+0.03
    return lo, hi

def sim_to_bank_selfLOO():
    # leave-one-out self similarity trong poison bank
    G = POIS_BANK @ POIS_BANK.T
    np.fill_diagonal(G, -1.0)
    return G.max(1)

def remap_confidence(s, p, R):
    if s < R["MIN_KEEP"]: return 0.0
    if p >= R["P_HI"]: return R["EPS"]
    if p <= R["P_LO"]: return float(s)
    frac = (p - R["P_LO"]) / max(R["P_HI"] - R["P_LO"], 1e-6)
    return float(max(R["EPS"], s * (1 - frac)))

# Pass 1: cache candidates + s_diff/s_geo/s_dino cho moi anh (tinh 1 lan)
cache = {}
dino_calib_samples = []
for img_path in tqdm(test_files, desc="Pass1 candidates"):
    stem = img_path.stem
    im = load_image(img_path)
    co = poisoned_predictor(im)["instances"].to("cpu")
    cb = co.pred_boxes.tensor.numpy(); cs = co.scores.numpy()
    do = depoisoned_predictor(im)["instances"].to("cpu")
    db = do.pred_boxes.tensor.numpy(); ds = do.scores.numpy()
    if len(dino_calib_samples) < 200: dino_calib_samples.append((im, db, ds))
    if len(cb)==0:
        cache[stem] = (cb, cs, np.zeros(0,np.float32), np.zeros(0,np.float32), None); continue
    s_diff = np.ones(len(cb), np.float32)
    if len(db)>0:
        ious = iou_matrix(cb, db); j = ious.argmax(1); bi = ious[np.arange(len(cb)), j]
        for i in range(len(cb)):
            if bi[i] >= 0.5: s_diff[i] = 1.0 - (ds[j[i]] / max(cs[i],1e-6))
    s_geo = geometry_scores(cb)
    cache[stem] = (cb, cs, np.clip(s_diff,0,1), s_geo, im)

SIM_LO, SIM_HI = _dino_calibrate(dino_calib_samples) if DINO_OK else (0.85,0.94)
print(f"DINOv2 sim calib: SIM_LO(streak)={SIM_LO:.3f} SIM_HI(poison)={SIM_HI:.3f}")
def dino_prob(sim): return np.clip((sim-SIM_LO)/(SIM_HI-SIM_LO+1e-6),0,1).astype(np.float32)

def build_preds(use_dino, extra_floor=0.0):
    preds={}
    for stem,(cb,cs,s_diff,s_geo,im) in cache.items():
        if len(cb)==0: preds[stem]=(cb,cs); continue
        p = W_DIFF*s_diff + W_GEO*s_geo
        if use_dino and im is not None:
            p = p + W_DINO*dino_prob(sim_to_bank(embed_boxes(im, cb)))
        p = np.clip(p,0,1)
        nc = np.array([remap_confidence(s,pp,R) for s,pp in zip(cs,p)], np.float32)
        if extra_floor>0: nc[cs<extra_floor]=0.0
        keep = nc>0.0
        eps_ids=np.where(nc<=R["EPS"]+1e-6)[0]; strong=np.where(nc>0.20)[0]
        if len(eps_ids) and len(strong):
            ov=iou_matrix(cb[eps_ids], cb[strong]).max(1); keep[eps_ids[ov>=0.20]]=False
        preds[stem]=(cb[keep], nc[keep])
    return preds

def dpi(preds): 
    t=sum(len(s[1]) for s in preds.values()); return t, t/len(test_files)

# Anchor thuan 229 (khong dino, khong extra floor)
preds_anchor = build_preds(use_dino=False, extra_floor=0.0)
ta,da = dpi(preds_anchor); print(f"ANCHOR 229: dets={ta} ({da:.3f}/img)")

# Dino variant + KHOA MAT DO: quet extra_floor de ve gan TARGET_DPI
print("Density lock sweep (dino variant):")
best_floor, best_gap, preds_dino = 0.0, 1e9, None
for fl in [0.0,0.20,0.25,0.30,0.35,0.40,0.45,0.50]:
    pv = build_preds(use_dino=True, extra_floor=fl)
    t,d = dpi(pv); gap=abs(d-TARGET_DPI)
    print(f"  floor={fl:.2f} -> dets={t} ({d:.3f}/img)")
    if gap<best_gap: best_gap,best_floor,preds_dino = gap,fl,pv
td,dd = dpi(preds_dino)
print(f"-> chon extra_floor={best_floor:.2f} | DINO variant dets={td} ({dd:.3f}/img) (target {TARGET_DPI}/img)")


Running inference on 2000 test images...


Pass1 candidates: 100%|██████████| 2000/2000 [06:58<00:00,  4.78it/s]


DINOv2 sim calib: SIM_LO(streak)=0.878 SIM_HI(poison)=0.908
ANCHOR 229: dets=2585 (1.292/img)
Density lock sweep (dino variant):
  floor=0.00 -> dets=2586 (1.293/img)
  floor=0.20 -> dets=2586 (1.293/img)
  floor=0.25 -> dets=2324 (1.162/img)
  floor=0.30 -> dets=2152 (1.076/img)
  floor=0.35 -> dets=1977 (0.989/img)
  floor=0.40 -> dets=1793 (0.896/img)
  floor=0.45 -> dets=1633 (0.817/img)
  floor=0.50 -> dets=1449 (0.725/img)
-> chon extra_floor=0.50 | DINO variant dets=1449 (0.725/img) (target 0.7/img)


## 11. Write Submission File


In [12]:
sample = pd.read_csv(SAMPLE_SUB, dtype={"image_id": str})

def format_preds(bx, sc):
    parts=[]
    for (x1,y1,x2,y2),s in zip(bx,sc):
        x1,y1=float(np.clip(x1,0,IMG_W)),float(np.clip(y1,0,IMG_H))
        x2,y2=float(np.clip(x2,0,IMG_W)),float(np.clip(y2,0,IMG_H))
        w,h=x2-x1,y2-y1
        if w>0 and h>0 and s>0: parts+=[f"{s:.6f}",f"{x1:.2f}",f"{y1:.2f}",f"{w:.2f}",f"{h:.2f}"]
    return " ".join(parts) or " "

def write_sub(preds, path):
    d=sample.copy()
    d["prediction_string"]=d["image_id"].map(lambda i: format_preds(*preds.get(str(i),(np.zeros((0,4)),np.zeros(0)))))
    d.to_csv(path,index=False)
    tot=sum(len(s[1]) for s in preds.values())
    print(f"saved {path} | {len(d)} rows | {tot} preds ({tot/len(d):.3f}/img)")

write_sub(preds_anchor, "/kaggle/working/submission_229.csv")   # anchor thuan 229
write_sub(preds_dino,   "/kaggle/working/submission.csv")       # 229 + DINOv2 drop, density-locked
print("DEFAULT submission.csv = 229 + DINOv2 drop (density-locked ~0.7/img) | anchor = submission_229.csv")
print("Ky luat: neu DINO variant density lech xa 0.7 hoac fired tren unlearn khong giam -> nop anchor 229 truoc.")


saved /kaggle/working/submission_229.csv | 2000 rows | 2585 preds (1.292/img)
saved /kaggle/working/submission.csv | 2000 rows | 1449 preds (0.725/img)
DEFAULT submission.csv = 229 + DINOv2 drop (density-locked ~0.7/img) | anchor = submission_229.csv
Ky luat: neu DINO variant density lech xa 0.7 hoac fired tren unlearn khong giam -> nop anchor 229 truoc.
